In [2]:
#!/usr/bin/env python3
"""
test_results.py

Evaluates the results of an Hy2DL training run by reading its
testing_metrics.zarr and testing_results.zarr stores, cleaning out
broken gauges (e.g. -inf NSE from zero-variance observed series),
printing a summary, and saving a clean per-basin CSV.

Run this from inside the run's output folder
(e.g. .../results/LSTM_CAMELS_DE_1h_benchmark_seed_100/), or pass
--run_dir pointing to it.

Usage:
    python test_results.py
    python test_results.py --run_dir .
    python test_results.py --run_dir /path/to/results/LSTM_..._seed_100 --top_n 15
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import xarray as xr


def load_metrics(run_dir: Path):
    path = run_dir / "testing_metrics.zarr"
    if not path.exists():
        print(f"[!] testing_metrics.zarr not found at: {path}")
        return None
    try:
        ds = xr.open_zarr(path)
    except Exception as e:
        print(f"[!] Failed to open testing_metrics.zarr: {e}")
        return None

    print("=== testing_metrics.zarr structure ===")
    print(ds)
    return ds


def load_results(run_dir: Path):
    path = run_dir / "testing_results.zarr"
    if not path.exists():
        print(f"[!] testing_results.zarr not found at: {path}")
        return None
    try:
        ds = xr.open_zarr(path)
        print("\n=== testing_results.zarr structure ===")
        print(ds)
        return ds
    except Exception as e:
        print(f"[!] Failed to open testing_results.zarr: {e}")
        return None


def summarize(ds_metrics, metric_name: str, feature_names, top_n: int, run_dir: Path):
    """Handles the standard Hy2DL layout: one DataArray with dims
    (metric, gauge_id, feature), where `metric` holds e.g. 'nse'/'rmse'."""
    da = ds_metrics["__xarray_dataarray_variable__"]

    if metric_name not in da["metric"].values:
        print(f"\n[!] Metric '{metric_name}' not found. Available: {list(da['metric'].values)}")
        return

    metric_da = da.sel(metric=metric_name)
    df = metric_da.to_pandas()  # rows = gauge_id, columns = feature

    if feature_names is None:
        feature_names = list(df.columns)

    # --- Identify and report broken gauges (inf / -inf) before they poison stats ---
    is_bad = np.isinf(df).any(axis=1)
    bad_gauges = df.index[is_bad].tolist()

    print(f"\n=== {metric_name.upper()} summary ===")
    print(f"Total gauges: {df.shape[0]}")
    print(f"Gauges with inf/-inf {metric_name.upper()} (excluded from stats below): {len(bad_gauges)}")
    if bad_gauges:
        print(f"  -> {bad_gauges}")

    df_clean = df.replace([np.inf, -np.inf], np.nan)

    print("\nDescribe (cleaned):")
    print(df_clean.describe())

    print(f"\nMedian {metric_name.upper()} per feature:")
    print(df_clean.median())

    print(f"\nMean {metric_name.upper()} per feature:")
    print(df_clean.mean())

    for feature in feature_names:
        if feature not in df_clean.columns:
            continue
        print(f"\n--- Top {top_n} basins by {metric_name.upper()} ({feature}) ---")
        print(df_clean[feature].dropna().sort_values(ascending=False).head(top_n))
        print(f"\n--- Bottom {top_n} basins by {metric_name.upper()} ({feature}) ---")
        print(df_clean[feature].dropna().sort_values(ascending=True).head(top_n))

    out_csv = run_dir / f"{metric_name}_summary_clean.csv"
    df_clean.to_csv(out_csv)
    print(f"\n[+] Saved cleaned {metric_name.upper()} table to: {out_csv.resolve()}")


def main():
    parser = argparse.ArgumentParser(description="Summarize and clean Hy2DL testing results.")
    parser.add_argument("--run_dir", type=str, default=".", help="Path to the run's output folder.")
    parser.add_argument("--top_n", type=int, default=15, help="Number of best/worst basins to show.")
    parser.add_argument(
        "--metrics",
        type=str,
        nargs="+",
        default=["nse", "rmse"],
        help="Which metrics to summarize (must match names inside the 'metric' dimension).",
    )
    # parse_known_args so this also runs fine via %run inside a Jupyter cell
    args, _unknown = parser.parse_known_args()

    run_dir = Path(args.run_dir).resolve()
    if not run_dir.exists():
        print(f"[!] Run directory does not exist: {run_dir}")
        sys.exit(1)

    print(f"Run directory: {run_dir}\n")

    ds_metrics = load_metrics(run_dir)
    load_results(run_dir)  # just prints structure; too large to load fully into a dataframe

    if ds_metrics is None:
        sys.exit(1)

    feature_names = None
    if "feature" in ds_metrics.coords:
        feature_names = list(ds_metrics["feature"].values)
        print(f"\nFeatures found: {feature_names}")

    for metric_name in args.metrics:
        summarize(ds_metrics, metric_name, feature_names, args.top_n, run_dir)


if __name__ == "__main__":
    main()

Run directory: /pfs/data6/home/ka/ka_iwu/ka_yh2352/Hy2DL-Testing/results/LSTM_CAMELS_DE_1h_benchmark_seed_200

=== testing_metrics.zarr structure ===
<xarray.Dataset> Size: 13kB
Dimensions:                        (metric: 2, gauge_id: 221, feature: 3)
Coordinates:
  * metric                         (metric) <U4 32B 'nse' 'rmse'
  * gauge_id                       (gauge_id) <U8 7kB 'DE110000' ... 'DE112570'
  * feature                        (feature) <U19 228B 'precipitation_mean' ....
Data variables:
    __xarray_dataarray_variable__  (metric, gauge_id, feature) float32 5kB dask.array<chunksize=(2, 221, 3), meta=np.ndarray>

=== testing_results.zarr structure ===
<xarray.Dataset> Size: 233MB
Dimensions:   (gauge_id: 221, date: 43847, feature: 3)
Coordinates:
  * gauge_id  (gauge_id) object 2kB 'DE110000' 'DE110010' ... 'DE112570'
  * date      (date) datetime64[ns] 351kB 2020-01-01T01:00:00 ... 2024-12-31T...
  * feature   (feature) <U19 228B 'precipitation_mean' ... 'precipitation_st